In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
servo_data = fetch_ucirepo(id=87)

In [ ]:
servo_df = servo_data.data.original.iloc[:,2:]

In [ ]:
servo_df

In [ ]:
servo_features = servo_df.iloc[:,:-1]
servo_target = servo_df.iloc[:,-1]

In [ ]:
mms = MinMaxScaler().set_output(transform='pandas')
servo_features = mms.fit_transform(servo_features)

## Manual Linear Regression via Gradient Descent

We learn weights **w** (for pgain, vgain) and a bias **b** by minimising
the Mean Squared Error (MSE) cost function:

$$J(\mathbf{w}) = \frac{1}{2m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)^2$$

The **vectorised** gradient descent update rules are:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} X^\top (\hat{\mathbf{y}} - \mathbf{y}), \quad \frac{\partial J}{\partial b} = \frac{1}{m} \mathbf{1}^\top(\hat{\mathbf{y}} - \mathbf{y})$$

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \frac{\partial J}{\partial \mathbf{w}}, \quad b \leftarrow b - \alpha \cdot \frac{\partial J}{\partial b}$$

> **Why NumPy?** The loop-based version iterated over every sample and feature in pure Python. Replacing this with `X @ w` (a single BLAS matrix-multiply) eliminates all Python loop overhead.

In [ ]:
# Convert to NumPy arrays once — all math runs in C/BLAS after this
X = servo_features.to_numpy()              # shape (167, 2)
y = servo_target.to_numpy().astype(float)  # shape (167,)

n, feat_count = X.shape                    # 167 samples, 2 features

w      = np.zeros(feat_count)              # shape (2,)
b      = 0.0
ALPHA  = 0.001
EPOCHS = 10_000


def predict(w: np.ndarray, b: float) -> np.ndarray:
    """
    Compute predictions for all samples at once.
    X @ w is a single BLAS matrix-multiply replacing the old double for-loop.
    Returns shape (n,).
    """
    return X @ w + b


def cost(w: np.ndarray, b: float) -> float:
    """MSE cost (vectorised): J = (1/2m) * ||y_hat - y||^2"""
    residual = predict(w, b) - y           # shape (n,)
    return np.dot(residual, residual) / (2 * n)


def newWeights(w: np.ndarray, b: float):
    """
    One vectorised gradient descent step.

    residual = y_hat - y                  shape (n,)
    dw       = (1/m) * X^T @ residual     shape (2,)  -- one BLAS call
    db       = mean(residual)             scalar
    """
    residual = predict(w, b) - y           # (n,)
    dw = (X.T @ residual) / n             # (2,)
    db = np.mean(residual)
    return w - ALPHA * dw, b - ALPHA * db

## Training Loop

In [ ]:
cost_history = []

for epoch in range(EPOCHS):
    w, b = newWeights(w, b)
    if epoch % 1000 == 0:
        c = cost(w, b)
        cost_history.append((epoch, c))
        print(f"Epoch {epoch:>6}  |  Cost: {c:.6f}  |  Weights: {np.round(w, 4).tolist()}, bias: {b:.4f}")

print(f"\nFinal weights — pgain: {w[0]:.4f}, vgain: {w[1]:.4f}, bias: {b:.4f}")

## Evaluation

In [ ]:
# Generate predictions for every sample (one matrix multiply)
predictions = predict(w, b)                # shape (n,)

# R² score
ss_res = np.sum((predictions - y) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot)

# Mean Absolute Error
mae = np.mean(np.abs(predictions - y))

print(f"Final MSE  : {cost(w, b) * 2:.6f}")
print(f"Final MAE  : {mae:.6f}")
print(f"R² Score   : {r2:.6f}")

## Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Left: predicted vs actual scatter ---
axes[0].scatter(y, predictions, alpha=0.6, edgecolors='k', linewidths=0.4)
lims = [min(y.min(), predictions.min()), max(y.max(), predictions.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Predicted vs Actual  (R² = {r2:.3f})')
axes[0].legend()

# --- Right: cost over epochs ---
epochs_logged, costs_logged = zip(*cost_history)
axes[1].plot(epochs_logged, costs_logged, marker='o', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cost (MSE / 2)')
axes[1].set_title('Cost vs Epoch')

plt.tight_layout()
plt.show()